# Notebook 05: FastAPI Backend

Wraps the full  pipeline into a REST API and logs every prediction to Supabase (PostgreSQL).

| Cell | What happens |
|---|---|
| 1 | Install dependencies |
| 2 | Load secrets + configure paths |
| 3 | Load model (HF) + FAISS index (Kaggle dataset) |
| 4 | Supabase — create predictions table |
| 5 | Full pipeline functions (classify → SHAP → risk → RAG) |
| 6 | FastAPI app — /health, /predict, /stats, /history |
| 7 | Run server + test all endpoints with httpx |
| 8 | Save notebook + push to GitHub |

**Inputs**: `Bhoovika/scamsense-xlmroberta` (HF), `/kaggle/input/scamsense-rag/` (FAISS index)  
**Output**: Logged predictions in Supabase, tested API 

## Cell 1 — Install dependencies

In [1]:
import subprocess, sys, importlib

PACKAGES = [
    ("numpy",                "numpy<2"),
    ("fastapi",              "fastapi==0.111.0"),
    ("uvicorn",              "uvicorn==0.29.0"),
    ("httpx",                "httpx==0.27.0"),
    ("psycopg2",             "psycopg2-binary==2.9.9"),
    ("sentence_transformers","sentence-transformers==3.0.1"),
    ("langdetect",           "langdetect"),
    ("langgraph",            "langgraph"),
    ("shap",                 "shap"),
    ("pydantic",             "pydantic==2.7.1"),
    ("anyio",                "anyio==4.3.0"),
    ("faiss",                "faiss-cpu"),          
    ("transformers",         "transformers"),        
    ("torch",                "torch"),               
    ("pandas",               "pandas"),              
]

to_install = []
for import_name, pip_spec in PACKAGES:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "?")
        print(f"  already installed  {import_name:<28} {ver}")
    except ImportError:
        print(f"   will install       {pip_spec}")
        to_install.append(pip_spec)

print(f"\n{len(to_install)} package(s) to install, "
      f"{len(PACKAGES)-len(to_install)} already present.\n")

if to_install:
    print("── Installing ────────────────────────────────────────────")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade"] + to_install,
        capture_output=True, text=True
    )
    for line in (result.stdout + result.stderr).splitlines():
        line = line.strip()
        if any(kw in line for kw in
               ("Collecting", "Downloading", "Installing", "Successfully",
                "already satisfied", "ERROR", "WARNING", "━")):
            print(" ", line)
    if result.returncode != 0:
        print("\n pip exited with errors — check above")
    else:
        print("\n All installs done")
else:
    print(" Nothing to install — all packages already present")

  already installed  numpy                        2.0.2
  already installed  fastapi                      0.136.1
  already installed  uvicorn                      0.46.0
  already installed  httpx                        0.28.1
  already installed  psycopg2                     2.9.12 (dt dec pq3 ext lo64)
  already installed  sentence_transformers        5.4.1
   will install       langdetect
  already installed  langgraph                    ?
  already installed  shap                         0.51.0
  already installed  pydantic                     2.12.3
  already installed  anyio                        ?
   will install       faiss-cpu
  already installed  transformers                 5.0.0
  already installed  torch                        2.10.0+cu128
  already installed  pandas                       2.3.3

2 package(s) to install, 13 already present.

── Installing ────────────────────────────────────────────
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.2 MB/s eta 0:0

## Cell 2 — Secrets + paths

In [3]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

_s = UserSecretsClient()
HF_TOKEN           = _s.get_secret("HF_TOKEN")
GITHUB_TOKEN       = _s.get_secret("GITHUB_TOKEN")
SUPABASE_URL       = _s.get_secret("SUPABASE_PROJECT_URL")   
SUPABASE_ANON_KEY  = _s.get_secret("SUPABASE_ANON_KEY")

# Paths
WORKING_DIR = Path("/kaggle/working")
RAG_DIR     = Path("/kaggle/input/datasets/bhoovika/scamsense-rag")   
OUTPUT_DIR  = WORKING_DIR / "scamsense_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Secrets loaded")
print(f"RAG dataset : {RAG_DIR}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"FAISS index : {(RAG_DIR / 'spf_faiss.index').exists()}")
print(f"SPF corpus  : {(RAG_DIR / 'spf_corpus.json').exists()}")

from urllib.parse import urlparse, urlunparse, quote


Secrets loaded
RAG dataset : /kaggle/input/datasets/bhoovika/scamsense-rag
Output dir  : /kaggle/working/scamsense_output
FAISS index : True
SPF corpus  : True


## Cell 3 — Load model + FAISS index

In [4]:
import json, re, torch, faiss, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import shap
from langdetect import detect, LangDetectException

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Classifier ────────────────────────────────────────────────────────────────
HF_MODEL_ID   = "Bhoovika/scamsense-xlmroberta"
clf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)
clf_model     = AutoModelForSequenceClassification.from_pretrained(
    HF_MODEL_ID, token=HF_TOKEN
).to(DEVICE)
clf_model.eval()
print("Classifier loaded")

# ── FAISS index + SPF corpus ──────────────────────────────────────────────────
faiss_index = faiss.read_index(str(RAG_DIR / "spf_faiss.index"))
with open(RAG_DIR / "spf_corpus.json") as f:
    SPF_CORPUS = json.load(f)
print(f"FAISS index loaded   ({faiss_index.ntotal} vectors)")
print(f"SPF corpus loaded    ({len(SPF_CORPUS)} chunks)")

# ── Embedding model for RAG ───────────────────────────────────────────────────
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Embedder loaded")

Device: cuda


config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedder loaded


## Cell 4 — Supabase: create predictions table

In [5]:
import httpx
from psycopg2.extras import RealDictCursor
from datetime import datetime, timezone

SUPABASE_HEADERS = {
    "apikey": SUPABASE_ANON_KEY,
    "Authorization": f"Bearer {SUPABASE_ANON_KEY}",
    "Content-Type": "application/json",
    "Prefer": "return=representation"
}

def supabase_insert(record: dict):
    r = httpx.post(
        f"{SUPABASE_URL}/rest/v1/predictions",
        headers=SUPABASE_HEADERS,
        json=record,
        timeout=15
    )
    r.raise_for_status()
    return r.json()

def supabase_select(query_params: dict = None, limit: int = 20):
    params = {"limit": limit, "order": "created_at.desc"}
    if query_params:
        params.update(query_params)
    r = httpx.get(
        f"{SUPABASE_URL}/rest/v1/predictions",
        headers=SUPABASE_HEADERS,
        params=params,
        timeout=15
    )
    r.raise_for_status()
    return r.json()


test = supabase_select(limit=1)
print("Supabase REST connected")
print("Table 'predictions' ready")

Supabase REST connected
Table 'predictions' ready


## Cell 5 — Full pipeline functions

In [6]:
# All pipeline functions 
LABEL_MAP = {0: "ham", 1: "scam"}

SINGLISH_RE = re.compile(
    r"\blah\b|\bleh\b|\bsia\b|\blor\b|\bwah\b|\baiyo\b|\bdun\b|\bwan\b|\bone\b",
    re.IGNORECASE
)

SPF_TAX = {
    "investment":    {"risk_score": 90},
    "impersonation": {"risk_score": 88},
    "job":           {"risk_score": 75},
    "phishing":      {"risk_score": 70},
    "ecommerce":     {"risk_score": 60},
    "loan":          {"risk_score": 65},
    "romance":       {"risk_score": 72},
    "unknown":       {"risk_score": 50},
}

SPF_KW = {
    "investment" : [r"invest", r"return", r"profit", r"crypto", r"trading",
                    r"guaranteed", r"bitcoin", r"ethereum", r"wallet", r"roi"],
    "impersonation": [r"police", r"spf", r"officer", r"arrest", r"investigation",
                      r"money laundering", r"safety account", r"警察", r"公安"],
    "job"        : [r"job", r"work from home", r"part.?time", r"salary",
                    r"earn", r"recruit", r"registration fee", r"kerja"],
    "phishing"   : [r"click", r"verify", r"suspend", r"otp", r"login",
                    r"http", r"\.xyz", r"bank", r"password", r"singpass"],
    "ecommerce"  : [r"sell", r"cheap", r"carousell", r"shopee", r"paynow first",
                    r"ship", r"legit", r"transfer first"],
    "romance"    : [r"love", r"relationship", r"dating", r"overseas",
                    r"send money", r"stranded", r"emergency"],
}

#  Language detection 
def detect_language(text: str) -> str:
    hits = len(SINGLISH_RE.findall(text))
    if len(text.split()) > 0 and hits / len(text.split()) >= 0.08:
        return "singlish"
    try:
        d = detect(text)
        return {"en": "en", "ms": "ms", "id": "ms", "ta": "ta",
                "zh-cn": "zh", "zh-tw": "zh", "zh": "zh"}.get(d, "en")
    except LangDetectException:
        return "en"

#  Classifier 
def run_classifier(text: str) -> dict:
    inputs = clf_tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=128, padding=True).to(DEVICE)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred  = int(probs.argmax())
    return {
        "label":      LABEL_MAP[pred],
        "confidence": round(float(probs[pred]), 4),
        "scam_prob":  round(float(probs[1]), 4),
    }

#  Scam type 
def get_scam_type(text: str) -> str:
    tl = text.lower()
    scores = {t: sum(1 for kw in kws if re.search(kw, tl))
              for t, kws in SPF_KW.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "unknown"

#  Risk scoring 
def get_risk(scam_type: str, scam_prob: float):
    base  = SPF_TAX.get(scam_type, SPF_TAX["unknown"])["risk_score"]
    score = max(10, min(100, int(base * scam_prob)))
    tier  = ("CRITICAL" if score >= 85 else "HIGH" if score >= 65
             else "MEDIUM" if score >= 40 else "LOW")
    return score, tier

#  SHAP 
def _shap_predict(texts):
    if isinstance(texts, str): texts = [texts]
    inputs = clf_tokenizer(list(texts), return_tensors="pt",
                           truncation=True, max_length=128, padding=True).to(DEVICE)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

_masker    = shap.maskers.Text(clf_tokenizer)
_explainer = shap.Explainer(_shap_predict, _masker, output_names=["ham", "scam"])

def get_top_shap(text: str, n: int = 5) -> list:
    sv = _explainer([text])
    pairs = [
        {"token": t, "shap_value": round(float(v), 4)}
        for t, v in zip(sv.data[0], sv.values[0, :, 1])
        if t not in ["", "▁", "<s>", "</s>", "<pad>"]
    ]
    return sorted(pairs, key=lambda x: x["shap_value"], reverse=True)[:n]

#  RAG retrieval 
def retrieve_spf(query: str, k: int = 3) -> list:
    vec = embedder.encode([query], normalize_embeddings=True).astype("float32")
    _, idxs = faiss_index.search(vec, k)
    return [SPF_CORPUS[i] for i in idxs[0] if i < len(SPF_CORPUS)]

#  Grounded explanation 
RISK_ADVICE = {
    "CRITICAL": "Do NOT transfer any money. Report immediately to SPF at www.police.gov.sg/iwitness or call 1800-255-0000.",
    "HIGH":     "Exercise extreme caution. Verify independently before taking any action. Report suspected scams to SPF.",
    "MEDIUM":   "Be cautious. Do not share personal information or make payments without verification.",
    "LOW":      "Stay alert. If something feels off, trust your instincts and verify through official channels.",
    "NONE":     "This message appears legitimate. Always stay vigilant against scams.",
}

def generate_grounded_explanation(
    message, label, confidence, language, scam_type,
    risk_tier, risk_score, top_features
) -> dict:
    passages = retrieve_spf(message, k=3)

    if label == "ham":
        explanation = (
            f"ScamSense classified this message as LEGITIMATE with "
            f"{confidence:.1%} confidence (language: {language}). "
            f"No scam indicators were detected. {RISK_ADVICE['NONE']}"
        )
        return {"explanation": explanation, "sources": [], "retrieved": []}

    # Build feature string
    feat_str = ""
    if top_features:
        tokens = [f["token"] for f in top_features[:3]]
        feat_str = f" Key scam indicators: {', '.join(tokens)}."

    # Pull strongest SPF passage
    spf_text  = passages[0]["text"] if passages else ""
    spf_topic = passages[0]["topic"] if passages else ""
    spf_page  = passages[0].get("source_page", "?") if passages else "?"

    explanation = (
        f"⚠️ SCAM DETECTED — {risk_tier} risk (score: {risk_score}/100)\n\n"
        f"ScamSense classified this {language}-language message as a "
        f"{scam_type} scam with {confidence:.1%} confidence.{feat_str}\n\n"
        f"📋 SPF Advisory ({spf_topic}, p.{spf_page}):\n"
        f"{spf_text[:300]}...\n\n"
        f"🛡️ What to do: {RISK_ADVICE[risk_tier]}"
    )

    sources = [
        f"SPF 2025 Annual Scams Brief — {p['topic']} (p.{p.get('source_page','?')})"
        for p in passages
    ]

    return {"explanation": explanation, "sources": sources, "retrieved": passages}


# ── Supabase logging ──────────────────────────────────────────────────────────
def supabase_insert(record: dict):
    r = httpx.post(
        f"{SUPABASE_URL}/rest/v1/predictions",
        headers=SUPABASE_HEADERS,
        json=record,
        timeout=15
    )
    r.raise_for_status()
    return r.json()

def log_prediction(record: dict):
    try:
        supabase_insert(record)
    except Exception as e:
        print(f" Supabase log failed: {e}")


# ── Master pipeline ───────────────────────────────────────────────────────────
def run_pipeline(message: str) -> dict:
    lang                         = detect_language(message)
    clf                          = run_classifier(message)
    label, conf, scam_prob       = clf["label"], clf["confidence"], clf["scam_prob"]
    top_features                 = get_top_shap(message) if label == "scam" else []
    scam_type                    = get_scam_type(message) if label == "scam" else None
    risk_score, risk_tier        = (get_risk(scam_type, scam_prob)
                                    if label == "scam" else (0, "NONE"))
    rag = generate_grounded_explanation(
        message=message, label=label, confidence=conf,
        language=lang, scam_type=scam_type or "unknown",
        risk_tier=risk_tier, risk_score=risk_score,
        top_features=top_features,
    )
    return {
        "message":      message,
        "language":     lang,
        "label":        label,
        "confidence":   conf,
        "scam_type":    scam_type,
        "risk_score":   risk_score,
        "risk_tier":    risk_tier,
        "top_features": top_features,
        "explanation":  rag["explanation"],
        "sources":      rag["sources"],
    }

print("Pipeline functions ready ")

Pipeline functions ready 


## Cell 6 — FastAPI app definition

In [7]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uvicorn

app = FastAPI(title="ScamSense API", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

#  Schemas 
class PredictRequest(BaseModel):
    message: str
    source_app: str = "api"

class StatsResponse(BaseModel):
    total_predictions: int
    total_scams: int
    total_ham: int
    scam_rate_pct: float
    by_risk_tier: dict
    by_language: dict
    by_scam_type: dict

#  /health 
@app.get("/health", tags=["System"])
def health():
    try:
        rows = supabase_select(limit=1)
        total = len(rows)
        db_status = "ok"
    except Exception as e:
        total = -1
        db_status = f"error: {e}"
    return {
        "status": "ok",
        "model": HF_MODEL_ID,
        "device": DEVICE,
        "db": db_status,
        "total_predictions": total,
    }

#  /predict 
@app.post("/predict", tags=["Inference"])
def predict(req: PredictRequest):
    result = run_pipeline(req.message)
    record = {
        "message":      result["message"],
        "language":     result["language"],
        "label":        result["label"],
        "confidence":   result["confidence"],
        "scam_type":    result.get("scam_type"),
        "risk_score":   result.get("risk_score"),
        "risk_tier":    result.get("risk_tier"),
        "explanation":  result.get("explanation"),
        "sources":      result.get("sources", []),
        "top_features": result.get("top_features", []),
        "source_app":   req.source_app,
    }
    try:
        inserted = supabase_insert(record)
        result["id"] = inserted[0].get("id") if inserted else None
    except Exception as e:
        print(f"[WARN] DB log failed: {e}")
        result["id"] = None
    return result

#  /stats 
@app.get("/stats", tags=["Analytics"], response_model=StatsResponse)
def stats():
    try:
        rows = supabase_select(limit=10000)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    total = len(rows)
    scams = sum(1 for r in rows if r.get("label") == "scam")
    ham   = total - scams

    by_tier, by_lang, by_type = {}, {}, {}
    for r in rows:
        tier  = r.get("risk_tier")  or "NONE"
        lang  = r.get("language")   or "unknown"
        stype = r.get("scam_type")  or "unknown"
        by_tier[tier]  = by_tier.get(tier, 0)  + 1
        by_lang[lang]  = by_lang.get(lang, 0)  + 1
        by_type[stype] = by_type.get(stype, 0) + 1

    scam_rate = round(scams / total * 100, 1) if total > 0 else 0.0
    return StatsResponse(
        total_predictions=total,
        total_scams=scams,
        total_ham=ham,
        scam_rate_pct=scam_rate,
        by_risk_tier=by_tier,
        by_language=by_lang,
        by_scam_type=by_type,
    )

#  /history 
@app.get("/history", tags=["Analytics"])
def history(limit: int = 20, label: Optional[str] = None):
    try:
        params = {"label": f"eq.{label}"} if label else {}
        rows = supabase_select(query_params=params, limit=limit)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

    keys = ["id","created_at","message","language","label",
            "confidence","scam_type","risk_tier","risk_score"]
    return {"count": len(rows), "predictions": [{k: r.get(k) for k in keys} for r in rows]}

print("FastAPI app defined")
print("Endpoints:")
print("  GET  /health   — liveness check")
print("  POST /predict  — full pipeline + Supabase log")
print("  GET  /stats    — aggregate analytics")
print("  GET  /history  — recent predictions")

FastAPI app defined
Endpoints:
  GET  /health   — liveness check
  POST /predict  — full pipeline + Supabase log
  GET  /stats    — aggregate analytics
  GET  /history  — recent predictions


## Cell 7 — Run server + test all endpoints

In [8]:
from IPython.display import display
import threading, time, httpx, json, os

# Kill any existing server on port 8000
os.system("fuser -k 8000/tcp")
time.sleep(1)

API_BASE = "http://127.0.0.1:8000"

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

t = threading.Thread(target=_run, daemon=True)
t.start()
time.sleep(4)
print("Server running on", API_BASE)

API_BASE = "http://127.0.0.1:8000"

# Start uvicorn in a background thread
def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

t = threading.Thread(target=_run, daemon=True)
t.start()
time.sleep(4)   
print("Server running on", API_BASE)

#  Helper 
def pp(label, response):
    print(f"\n{'='*60}")
    print(f"  {label}  [{response.status_code}]")
    print('='*60)
    try:
        data = response.json()
        print(json.dumps(data, indent=2, ensure_ascii=False)[:1500])
    except:
        print(response.text[:500])

client = httpx.Client(base_url=API_BASE, timeout=120.0)

#  Test 1: Health 
pp("GET /health", client.get("/health"))

#  Test 2: Predict — English phishing 
pp("POST /predict — EN phishing", client.post("/predict", json={
    "message": "Your POSB account has been suspended. Verify at http://posb-secure.xyz",
    "source_app": "notebook_test"
}))

#  Test 3: Predict — Mandarin impersonation 
pp("POST /predict — ZH impersonation", client.post("/predict", json={
    "message": "您好，我是新加坡警察局侦探。您的账户涉嫌洗钱，请立即转账至安全账户。",
    "source_app": "notebook_test"
}))

#  Test 4: Predict — Singlish ecommerce 
pp("POST /predict — Singlish ecommerce", client.post("/predict", json={
    "message": "Selling iPhone 15 Pro Max $400 only lah! PayNow me first then I ship lor.",
    "source_app": "notebook_test"
}))

#  Test 5: Predict — Tamil investment 
pp("POST /predict — TA investment", client.post("/predict", json={
    "message": "உங்கள் முதலீட்டில் 300% லாபம் உறுதி. இப்போதே பதிவு செய்யுங்கள்!",
    "source_app": "notebook_test"
}))

# ── Test 6: Predict — Malay job scam ─────────────────────────────────────────
pp("POST /predict — MS job scam", client.post("/predict", json={
    "message": "Kerja mudah dari rumah! Pendapatan RM5,000 sebulan. Bayar yuran pendaftaran RM200.",
    "source_app": "notebook_test"
}))

#  Test 7: Predict — Ham 
pp("POST /predict — EN ham", client.post("/predict", json={
    "message": "Hi! Your dentist appointment is tomorrow 2pm at Raffles Dental. Reply to confirm.",
    "source_app": "notebook_test"
}))

#  Test 8: Stats 
pp("GET /stats", client.get("/stats"))

#  Test 9: History 
pp("GET /history?limit=5", client.get("/history?limit=5"))

client.close()
print("\n All endpoint tests complete")

Server running on http://127.0.0.1:8000


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


Server running on http://127.0.0.1:8000

  GET /health  [200]
{
  "status": "ok",
  "model": "Bhoovika/scamsense-xlmroberta",
  "device": "cuda",
  "db": "ok",
  "total_predictions": 1
}

  POST /predict — EN phishing  [200]
{
  "message": "Your POSB account has been suspended. Verify at http://posb-secure.xyz",
  "language": "en",
  "label": "scam",
  "confidence": 1.0,
  "scam_type": "phishing",
  "risk_score": 70,
  "risk_tier": "HIGH",
  "top_features": [
    {
      "token": "z",
      "shap_value": 0.1627
    },
    {
      "token": "POS",
      "shap_value": 0.16
    },
    {
      "token": "Your ",
      "shap_value": 0.0884
    },
    {
      "token": "suspend",
      "shap_value": 0.0812
    },
    {
      "token": "account ",
      "shap_value": 0.0699
    }
  ],
  "explanation": "⚠️ SCAM DETECTED — HIGH risk (score: 70/100)\n\nScamSense classified this en-language message as a phishing scam with 100.0% confidence. Key scam indicators: z, POS, Your .\n\n📋 SPF Advisory (Bank 

## Cell 8 — Export prediction log

In [9]:
import subprocess, pandas as pd

#  Export predictions to CSV via REST 
rows = supabase_select(limit=10000)
df = pd.DataFrame(rows)


cols = ["id","created_at","message","language","label","confidence",
        "scam_type","risk_score","risk_tier","sources","source_app"]
df = df[[c for c in cols if c in df.columns]]

csv_path = OUTPUT_DIR / "05_prediction_log.csv"
df.to_csv(csv_path, index=False)
print(f"Prediction log saved → {csv_path}  ({len(df)} rows)")
if not df.empty:
    print(df[["label","language","risk_tier","scam_type","confidence"]].to_string(index=False))
    

print("   API endpoints : /health  /predict  /stats  /history")
print("   DB            : Supabase REST API (persistent)")
print("   Logged rows   :", len(df))

Prediction log saved → /kaggle/working/scamsense_output/05_prediction_log.csv  (69 rows)
label language risk_tier     scam_type  confidence
  ham       en      NONE          None      0.9984
 scam       ms      HIGH           job      1.0000
 scam       ta    MEDIUM       unknown      1.0000
 scam singlish    MEDIUM     ecommerce      1.0000
 scam       zh  CRITICAL impersonation      0.9995
 scam       en      HIGH      phishing      1.0000
 scam       zh    MEDIUM       unknown      0.9995
 scam       zh  CRITICAL impersonation      0.9994
  ham       en      NONE          None      1.0000
 scam       ms      HIGH           job      1.0000
  ham       en      NONE          None      0.9203
 scam       ta    MEDIUM       unknown      1.0000
  ham       en      NONE          None      0.9978
  ham       en      NONE          None      0.9978
 scam       en      HIGH      phishing      1.0000
  ham       en      NONE          None      0.9203
 scam       ms      HIGH           job      

## Cell 9 — Telegram Bot (ScamSense Scout)

**Prerequisites before running:**
1. Create a bot via [@BotFather](https://t.me/BotFather) → `/newbot` → copy the token.
2. Add the token as a Kaggle secret named `TELEGRAM_BOT_TOKEN`.
3. The FastAPI server from Cell 7 must be running in the background thread.

In [10]:
# Install python-telegram-bot
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "python-telegram-bot==20.8"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("ERROR:", result.stderr[-400:])
else:
    print("python-telegram-bot==20.9 installed")

python-telegram-bot==20.9 installed


In [11]:
#  Compatibility patch 
import httpx as _httpx, functools

if not getattr(_httpx.AsyncClient, "_proxies_patched", False):
    _real_init = _httpx.AsyncClient.__init__
    @functools.wraps(_real_init)
    def _patched_init(self, *args, **kwargs):
        kwargs.pop("proxies", None)
        _real_init(self, *args, **kwargs)
    _httpx.AsyncClient.__init__ = _patched_init
    _httpx.AsyncClient._proxies_patched = True

print("httpx AsyncClient patched   (proxies kwarg suppressed)")

import asyncio, httpx, threading, time
from telegram import Update
from telegram.ext import (
    ApplicationBuilder, CommandHandler, MessageHandler,
    filters, ContextTypes
)
from kaggle_secrets import UserSecretsClient

#  Load token 
_s = UserSecretsClient()
TELEGRAM_TOKEN = _s.get_secret("TELEGRAM_BOT_TOKEN")

API_BASE = "http://127.0.0.1:8000"   # FastAPI server from Cell 7

#  Risk emoji map 
TIER_EMOJI = {
    "CRITICAL": "🔴",
    "HIGH":     "🟠",
    "MEDIUM":   "🟡",
    "LOW":      "🟢",
    "NONE":     "✅",
}

#  /start 
async def cmd_start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "👋 Hi! I'm *ScamSense Scout* — your personal scam checker.\n\n"
        "Just forward me any suspicious message (SMS, email text, WhatsApp) "
        "and I'll analyse it using Singapore Police Force scam intelligence.\n\n"
        "Commands:\n"
        "  /start — show this message\n"
        "  /help  — how to use the bot\n"
        "  /stats — scan statistics",
        parse_mode="Markdown"
    )

#  /help 
async def cmd_help(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "📋 *How to use ScamSense Scout*\n\n"
        "1. Copy or forward any suspicious message to this chat.\n"
        "2. I'll run it through the ScamSense AI pipeline.\n"
        "3. You'll get a risk rating, scam type, and advice.\n\n"
        "Supported languages: English, Singlish, Malay, Tamil, Mandarin.",
        parse_mode="Markdown"
    )

#  /stats 
async def cmd_stats(update: Update, context: ContextTypes.DEFAULT_TYPE):
    try:
        r = httpx.get(f"{API_BASE}/stats", timeout=15)
        r.raise_for_status()
        d = r.json()
        msg = (
            f"📊 *ScamSense Stats*\n\n"
            f"Total scans : {d['total_predictions']}\n"
            f"Scams found : {d['total_scams']}  ({d['scam_rate_pct']}%)\n"
            f"Legitimate  : {d['total_ham']}\n\n"
            f"*By risk tier:*\n"
        )
        for tier, count in d.get("by_risk_tier", {}).items():
            emoji = TIER_EMOJI.get(tier, "⚪")
            msg += f"  {emoji} {tier}: {count}\n"
        await update.message.reply_text(msg, parse_mode="Markdown")
    except Exception as e:
        await update.message.reply_text(f"⚠️ Could not fetch stats: {e}")

#  Helper: advice text per tier 
def _advice(tier: str) -> str:
    return {
        "CRITICAL": (
            "Do NOT transfer money. Report immediately to SPF at "
            "www.police.gov.sg/iwitness or call 1800-255-0000."
        ),
        "HIGH": (
            "Exercise extreme caution. Verify independently before "
            "any action. Report to SPF if confirmed."
        ),
        "MEDIUM": (
            "Be cautious. Do not share personal details or make "
            "payments without verifying the sender."
        ),
        "LOW": (
            "Stay alert. If something feels off, trust your instincts "
            "and verify through official channels."
        ),
        "NONE": "This message appears legitimate. Always stay vigilant.",
    }.get(tier, "Stay vigilant and report suspicious messages to SPF.")

#  Message handler (any text -> /predict) 
async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    text = update.message.text or update.message.caption or ""
    if not text.strip():
        await update.message.reply_text("Please send a text message to analyse.")
        return

    wait_msg = await update.message.reply_text("🔍 Analysing... please wait.")

    try:
        r = httpx.post(
            f"{API_BASE}/predict",
            json={"message": text, "source_app": "telegram"},
            timeout=120
        )
        r.raise_for_status()
        d = r.json()
    except Exception as e:
        await wait_msg.edit_text(f"❌ Analysis failed: {e}")
        return

    label      = d.get("label", "unknown")
    conf       = d.get("confidence", 0)
    lang       = d.get("language", "?")
    scam_type  = d.get("scam_type") or "—"
    risk_tier  = d.get("risk_tier", "NONE")
    risk_score = d.get("risk_score", 0)
    sources    = d.get("sources", [])
    emoji      = TIER_EMOJI.get(risk_tier, "⚪")

    if label == "ham":
        reply = (
            f"✅ *LEGITIMATE MESSAGE*\n\n"
            f"Confidence : {conf:.1%}\n"
            f"Language   : {lang}\n\n"
            f"No scam indicators detected. Stay vigilant — when in doubt, "
            f"verify through official channels."
        )
    else:
        sources_str = ""
        if sources:
            sources_str = (
                "\n\n📚 *SPF References:*\n"
                + "\n".join(f"• {s}" for s in sources[:2])
            )
        reply = (
            f"{emoji} *SCAM DETECTED — {risk_tier} RISK*\n\n"
            f"Confidence  : {conf:.1%}\n"
            f"Language    : {lang}\n"
            f"Scam type   : {scam_type}\n"
            f"Risk score  : {risk_score}/100"
            f"{sources_str}\n\n"
            f"🛡️ *What to do:*\n{_advice(risk_tier)}"
        )

    await wait_msg.edit_text(reply, parse_mode="Markdown")

#  Build and launch the bot 
def run_bot():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    app_bot = ApplicationBuilder().token(TELEGRAM_TOKEN).build()
    app_bot.add_handler(CommandHandler("start", cmd_start))
    app_bot.add_handler(CommandHandler("help",  cmd_help))
    app_bot.add_handler(CommandHandler("stats", cmd_stats))
    app_bot.add_handler(
        MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message)
    )

    print("🤖 ScamSense Scout bot starting (long-polling)...")
    app_bot.run_polling(stop_signals=None)

# Daemon thread — cell returns immediately; bot keeps polling in background
bot_thread = threading.Thread(target=run_bot, daemon=True)
bot_thread.start()

for i in range(10):
    time.sleep(1)
    if not bot_thread.is_alive():
        print(" Bot thread crashed — check the traceback above.")
        break
else:
    print("Bot is live. Open Telegram, find your bot, and send a message.")
    print("   Forward any suspicious text directly to the bot to scan it.")
    print("   To stop: restart the kernel or end the Kaggle session.")

httpx AsyncClient patched   (proxies kwarg suppressed)
🤖 ScamSense Scout bot starting (long-polling)...
Bot is live. Open Telegram, find your bot, and send a message.
   Forward any suspicious text directly to the bot to scan it.
   To stop: restart the kernel or end the Kaggle session.
